# 0. Setup (Layered architecture)

## Install Neccersary Packages

The following packages are needed for the program:

* `ipywidgets`: For UI
* `mne`: For data processing
* `numpy`
* `python-picard`: Specific method for ICA during Preprocessing



In [ ]:
%pip install ipywidgets
%pip install mne
%pip install numpy matplotlib scipy
%pip install python-picard
%pip install pandas 
%pip install xlrd openpyxl


## UI

Building the UI utilising `ipywidgets`

In [ ]:
from ipywidgets import interact, Dropdown, Text, VBox, HBox, Checkbox, IntSlider

fileUpload = Text(value='data.bdf', description='File Path:')
labelUpload = Text(value='participant_ratings.xls', description='File Path:')
downsampleOnOff = Checkbox(value=True, description='Downsample')
downsampleDropdown = Dropdown(options=['128', '256', '512'], value='128', description='Downsample Rate:')
bandpassOnOff = Checkbox(value=True, description='Bandpass Filter')
icaMethod = Dropdown(options=['fastica', 'infomax', 'picard'], value='picard', description='ICA Method:')
icaOnOff = Checkbox(value=False, description='ICA')
fastICA = Checkbox(value=True, description='ICA after Downsample')


ui = VBox([
    HBox([fileUpload, labelUpload]),
    HBox([downsampleOnOff, downsampleDropdown]),
    bandpassOnOff,
    HBox([icaOnOff,fastICA, icaMethod])
])

# things to add: FastICA (done after downsample)

In [ ]:
display(ui)

# 1. Classes

In [ ]:
import mne
import pandas as pd
from scipy.signal import decimate
from scipy.fft import rfft, rfftfreq
import numpy as np
from tqdm.auto import tqdm

RANDSEED = 52

### 1.1 Dataloader

In [ ]:
class Dataloader:
  def getData(self, fileUpload):
    raw = mne.io.read_raw_bdf(fileUpload, preload=True)
    #rawcopy = raw.copy().pick(mne.pick_types(raw.info, eeg=True)) This should work, but since DEAP doesnt label thier eeg channels as eeg, we have to brute force delete the non eeg channels (the next line)
    rawcopy = raw.drop_channels(raw.ch_names[32:47])

    events = mne.find_events(raw, stim_channel="Status", initial_event=True) #events comes as 3 columnes (sample Index, prev event code, event code), these are used for segmenting
    # rawcopy = rawcopy.pick('eeg') #removing the status column as it isnt needed anymore
    df = rawcopy.to_data_frame()
    return rawcopy, events, df

  def getLabels(self, labelUpload, particpant):
    df = pd.read_excel(labelUpload)
    valencedf = df[['Participant_id','Valence']]
    valence = valencedf[valencedf['Participant_id'] == particpant]    #just getting the first particpant while testing
    return valence


### 1.2 Preprocess

In [ ]:
class Preprocessing:
  def __init__(self, icaMethod, randSeed, nComponents = 0.999999):
    self.ica = mne.preprocessing.ICA(nComponents, random_state=randSeed, max_iter='auto', method = icaMethod.value)
    self.randSeed = randSeed
    self.icaMethod = icaMethod.value
  def preprocess(self, raw, downsampleTrue, downsamplerate, bandpassTrue, icaTrue, icaMethod,fastICA, events):
    rawCopy = raw.copy()
    rawCopy = rawCopy.filter(l_freq=1.5, h_freq=None) # low frequency drifts, gets rid of all freq under 1.5. this does effect the delta band...

    if(bandpassTrue == True): #bandpass True
      rawCopy = rawCopy.filter(l_freq=4, h_freq=45) #range 4 - 45 comes from the bandpass filter that deap use. The top range of the Gamma band is around 45Hz so we dont want anything higher
      if(icaTrue == True): #ICA AND bandpass True
        self.ica = mne.preprocessing.ICA(method=self.icaMethod, random_state=self.randSeed, max_iter='auto')
        self.ica.fit(rawCopy)
        eog_chs = [ch for ch in ['Fp1','Fp2'] if ch in rawCopy.ch_names]
        if eog_chs:
            eogArtifacts, _ = self.ica.find_bads_eog(rawCopy, ch_name=eog_chs)
        else:
          eogArtifacts = []  
          
        muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)
        exclude = (eogArtifacts or []) + (muscleArtifacts or [])
        exclude = [i for i in exclude if i < self.ica.n_components_]
        self.ica.exclude = exclude
        rawCopy = self.ica.apply(rawCopy.copy())
    else:
      if(icaTrue == True): #ICA AND bandpass True
        self.ica = mne.preprocessing.ICA(method=self.icaMethod, random_state=self.randSeed, max_iter='auto')
        self.ica.fit(rawCopy)
        eog_chs = [ch for ch in ['Fp1','Fp2'] if ch in rawCopy.ch_names]
        if eog_chs:
            eogArtifacts, _ = self.ica.find_bads_eog(rawCopy, ch_name=eog_chs)
        else:
          eogArtifacts = []  
          
        muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)
        exclude = (eogArtifacts or []) + (muscleArtifacts or [])
        exclude = [i for i in exclude if i < self.ica.n_components_]
        self.ica.exclude = exclude
        rawCopy = self.ica.apply(rawCopy.copy())

    if(downsampleTrue == True): #downsample
      rawCopy = rawCopy.resample(sfreq=int(downsamplerate))
      if(fastICA == True): #fast ICA
        self.ica = mne.preprocessing.ICA(method=self.icaMethod, random_state=self.randSeed, max_iter='auto')
        self.ica.fit(rawCopy)
        eog_chs = [ch for ch in ['Fp1','Fp2'] if ch in rawCopy.ch_names]
        if eog_chs:
            eogArtifacts, _ = self.ica.find_bads_eog(rawCopy, ch_name=eog_chs)
        else:
          eogArtifacts = []  
          
        muscleArtifacts, _ = self.ica.find_bads_muscle(rawCopy)
        exclude = (eogArtifacts or []) + (muscleArtifacts or [])
        exclude = [i for i in exclude if i < self.ica.n_components_]
        self.ica.exclude = exclude
        rawCopy = self.ica.apply(rawCopy.copy())

    df = rawCopy.to_data_frame()
    return rawCopy, df

### 1.3 Segmentation

In [ ]:
# class Segmentation:

#   def getEpochs(self, raw, eventId, labels):
#     tmin = -3.0         # start exactly at event onset
#     tmax = 60       # 60 seconds long (video length)
#     baselineTmin = -3.0
#     baselineTmax = 0.0
#     events = mne.find_events(raw, stim_channel="Status", initial_event=True)
#     sfreq = raw.info['sfreq']
#     n_samples = raw.n_times
#     first_event_sample = events[0, 0]
#     last_event_sample = events[-1, 0]

#     print("Start buffer (s):", first_event_sample / sfreq)
#     print("End buffer (s):", (n_samples - last_event_sample) / sfreq)
    
#     epochs = mne.Epochs(
#       raw,
#       events,
#       eventId,
#       tmin=tmin,
#       tmax=tmax,
#       reject= None,
#       flat= None,
#       baseline=(baselineTmin, baselineTmax),
#       preload=True,)
        
#     # epochs.plot(n_epochs=10, events=True)
#     epochs = epochs.drop_channels(["Status"])
#     epochsNumpy = epochs.get_data()


#     labels_array = np.array(labels)
#     labelsNotDropped = labels_array[epochs.events[:, 2] - 1]  # event IDs start at 1



#     return epochsNumpy,labelsNotDropped

In [ ]:
class Segmentation:
    def __init__(self, window_size=4.0, overlap=0.5):
        """
        window_size : length of each epoch in seconds
        overlap     : fraction overlap between windows (0.0–0.9)
        """
        self.window_size = window_size
        self.overlap = overlap

    def getEpochs(self, raw, events, eventId, labels):
        """
        raw      : preprocessed MNE Raw
        events   : DEAP event markers
        eventId  : event code to use for trial onset (e.g. {4: 'video'})
        labels   : np.array of labels (40 per participant)

        Returns:
          X_epochs : (n_epochs, n_channels, n_samples)
          y_labels : (n_epochs,)
        """
        sfreq = raw.info['sfreq']
        win_samp = int(self.window_size * sfreq)
        step_samp = int(win_samp * (1 - self.overlap))

        # Make 60s epochs first (just to anchor each video trial)
        tmin, tmax = 0, 60
        epochs = mne.Epochs(raw, events, event_id={'video': eventId}, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True)
        epochs = epochs.drop_channels(["Status"])
        data = epochs.get_data()   # shape (n_trials, n_channels, n_samples)

        X_segments, y_segments = [], []

        for trial_idx, trial_data in enumerate(data):
            label = labels[trial_idx]

            n_samples = trial_data.shape[-1]
            start = int(3 * sfreq)  # skip first 3s
            end = n_samples

            # Sliding windows inside this trial
            for s in range(start, end - win_samp + 1, step_samp):
                seg = trial_data[:, s:s+win_samp]  # (n_channels, win_samp)
                X_segments.append(seg)
                y_segments.append(label)

        return np.array(X_segments), np.array(y_segments)

### 1.4 Feature Extraction


In [ ]:
# class FeatureExtraction:
#     def __init__(self):
#         self.bands = {
#             'delta': (0.5, 4),
#             'theta': (4, 7),
#             'alpha': (7, 12),
#             'beta':  (12, 30),
#             'gamma': (30, 45) #gamma does go up to 80Hz, but our badpass filter cuts it off at 45 anyway 
#         }

#     def fftBandPowers(self, data, label, samplingRate, windowSizeSec=1.0, overlap=0.5):
#         step = int(windowSizeSec * samplingRate * (1 - overlap))
#         winLength = int(windowSizeSec * samplingRate)
#         nWindows = (data.shape[-1] - winLength) // step + 1
#         times = np.arange(nWindows) * step / samplingRate

#         X = []
#         for w in range(nWindows):
#             start = w * step
#             end = start + winLength
#             seg = data[..., start:end]
#             yf = np.abs(rfft(seg, axis=-1)) ** 2
#             xf = rfftfreq(winLength, 1 / samplingRate)

#             features = [np.sum(yf[..., (xf >= low) & (xf <= high)], axis=-1) for low, high in self.bands.values()]

#             X.append(np.stack(features, axis=-1))

#         if not X:  # no windows
#           return np.array([]), np.array([]), np.array([])

#         X = np.stack(X, axis=0)  # (windows, channels, bands)
#         labels_repeated = np.repeat(label, nWindows)
#         return times, X, labels_repeated

#     def extract(self, segments, labels, samplingRate, windowSizeSec=1.0, overlap=0.5):
#         allX, allTimes, allLabels = [], [], []
#         # if len(labels) != len(segments):
#           # raise ValueError(f"Labels length {len(labels)} does not match segments length {len(segments)}")
#         if len(segments) == 0:
#           return np.array([]), [], np.array([])
#         for i, seg in enumerate(segments):
#             times, bandPowers, repeated_labels = self.fftBandPowers(seg, labels[i], samplingRate, windowSizeSec, overlap)
#             if bandPowers.size == 0:
#               continue

#             bandPowers = bandPowers.reshape(bandPowers.shape[0], -1)  # flatten channels x bands
#             allX.append(bandPowers)
#             allTimes.extend(times)
#             allLabels.extend(repeated_labels)
#         if not allX:
#           return np.array([]), np.array([]), np.array([])  # No data to concatenate, return empty arrays or None as appropriate

#         allX = np.vstack(allX)
#         allTimes = np.array(allTimes)
#         allLabels = np.array(allLabels)
#         return allX, allTimes, allLabels


In [ ]:
class FeatureExtraction:
    def __init__(self):
        self.bands = {
            'delta': (0.5, 4),
            'theta': (4, 7),
            'alpha': (7, 12),
            'beta':  (12, 30),
            'gamma': (30, 45),
        }

    def fftBandPowers(self, seg, samplingRate):
        """
        seg: (n_channels, n_samples) single window
        """
        yf = np.abs(rfft(seg, axis=-1)) ** 2
        xf = rfftfreq(seg.shape[-1], 1 / samplingRate)

        features = [np.sum(yf[:, (xf >= low) & (xf <= high)], axis=-1)
                    for low, high in self.bands.values()]

        return np.concatenate(features, axis=-1)  # (n_channels * n_bands,)

    def extract(self, segments, labels, samplingRate):
        """
        segments : (n_epochs, n_channels, n_samples)
        labels   : (n_epochs,)
        """
        X = [self.fftBandPowers(seg, samplingRate) for seg in segments]
        X = np.stack(X, axis=0)
        return X, labels

#2. Pipeline

### 2.1 Constructors


In [ ]:
dl = Dataloader()
preProcessor = Preprocessing(randSeed = RANDSEED, nComponents = 0.999, icaMethod = icaMethod)
segmenter = Segmentation(window_size=4.0, overlap=0.5)
fE = FeatureExtraction()


### 2.2 Dataflow

In [ ]:
allX = []
allYLabels = []
participants  = []

for i in tqdm(range(1, 23)):  # participants 1 to 22
  fileUpload = f"s{i:02d}.bdf"
  raw, events, df = dl.getData(fileUpload) # events is shape (13761, 3), df is (1980928, 34)

  print(f"[Dataloader] Raw shape: {raw.get_data().shape}, sfreq: {raw.info['sfreq']}")
  labels = dl.getLabels(labelUpload.value, particpant = i) # labels are (particpant id, valence float score from 1 - 9) 40 values for 40 videos
  raw.plot()
  praw, df = preProcessor.preprocess(raw = raw,  #(495232, 33)
                                    downsampleTrue = downsampleOnOff.value,
                                    downsamplerate = downsampleDropdown.value,
                                    bandpassTrue = bandpassOnOff.value,
                                    icaTrue = icaOnOff.value,
                                    icaMethod = icaMethod,
                                    fastICA = fastICA.value,
                                    events = events)
  praw.plot()

  # epochs = segmenter.getEpochs(raw = praw, eventId= 4, labels = labels) #shape is like (num of epochs, channels, time sample for each channel)
  X_segments, y_segments = segmenter.getEpochs(
    raw=praw,
    events=events,
    eventId=4,
    labels=labels['Valence'].to_numpy()
  )

  # X, xTime, ylabels = fE.extract(segments = epochs, labels = labels['Valence'].to_numpy(), samplingRate = praw.info['sfreq'])
  X, ylabels = fE.extract(X_segments, y_segments, praw.info['sfreq'])

  allX.append(X)
  allYLabels.append(ylabels)
  participants.extend([i] * len(ylabels))

for i, X in enumerate(allX):
    print(f"[Concat Check] Participant {i}: {X.shape}")
allX = np.vstack(allX)
allYLabels = np.hstack(allYLabels)

In [ ]:
ylabels


In [ ]:
import pickle
with open("RawData22", 'rb') as f:
    rawX = pickle.load(allX)
    rawY = pickle.load(allYLabels)

In [ ]:
import pickle
with open("256ICAExcludeIncluded", 'wb') as f:
    pickle.dump(allX,f)
    pickle.dump(allYLabels,f)
    pickle.dump(participants,f)


In [ ]:
import pickle
with open("RawData22", 'wb') as f:
    pickle.dump(allX,f)
    pickle.dump(allYLabels,f)

In [ ]:
np.unique(y_discrete, return_counts=True)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# pca = PCA(n_components=0.95)  # keep 95% variance
# X_pca = pca.fit_transform(allX)

y_discrete = (allYLabels >= 5).astype(int)

lda = LDA(n_components=1)
X_lda = lda.fit_transform(allX, y_discrete)

X_lda, y_discrete = shuffle(X_lda, y_discrete, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_lda, y_discrete, test_size=0.2, random_state=42)

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

svm_clf = SVC(C=10, kernel='linear', gamma='scale')

svm_clf.fit(X_train, y_train)

y_pred = svm_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred) # binary classification
f1 = f1_score(y_test, y_pred) # for binary classification
report = classification_report(y_test, y_pred)

print(f"SVM: Accuracy: {acc:.4f}%, Recall :{recall:.4f}%, F1: {f1:.4f}%")
# print(f"Test Accuracy: {acc:.4f}%")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier # or KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # For data scaling
from sklearn.metrics import accuracy_score # For evaluating classification models

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(allX, y_discrete, test_size=0.3, random_state=42)

# Scale the data (important for distance-based algorithms like KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=2)

knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

kaccuracy = accuracy_score(y_test, y_pred)
krecall = recall_score(y_test, y_pred) # binary classification
kf1 = f1_score(y_test, y_pred) # for binary classification
kreport = classification_report(y_test, y_pred)

print(f"KNN: Accuracy: {kaccuracy:.4f}%, Recall :{krecall:.4f}%, F1: {kf1:.4f}%")
# print(f"Model Accuracy: {accuracy}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
np.unique(y_discrete, return_counts=True)

# Initialize
rf_clf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_depth=10,        # let trees grow fully
    random_state=42,
    n_jobs=-1              # use all CPU cores
)

# Train
rf_clf.fit(X_train, y_train)

# Predict
y_pred_rf = rf_clf.predict(X_test)

# Evaluate
rfaccuracy = accuracy_score(y_test, y_pred_rf)
rfrecall = recall_score(y_test, y_pred_rf) # binary classification
rff1 = f1_score(y_test, y_pred_rf) # for binary classification

print(f"RFC: Accuracy: {rfaccuracy:.4f}%, Recall :{rfrecall:.4f}%, F1: {rff1:.4f}%")

# 3. Results

# 4. Testing

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import numpy as np

# Assuming you already have:
# allX  -> concatenated feature matrix (n_epochs_total × n_features)
# allYLabels -> original continuous labels
# groups -> array of participant IDs, same length as allYLabels

y_discrete = (allYLabels >= 5).astype(int)

logo = LeaveOneGroupOut()

accuracies, recalls, f1s = [], [], []

for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
    X_train, X_test = allX[train_idx], allX[test_idx]
    y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

    # Standardize (fit only on train)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train model
    clf = SVC(C=10, kernel='rbf', gamma='scale')
    clf.fit(X_train, y_train)

    # Evaluate
    y_pred = clf.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))

# Report mean ± std across participants
print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

#Raw
#LOSO Accuracy: 0.2785 ± 0.3908
# LOSO Recall:   0.0429 ± 0.1966
# LOSO F1:       0.0441 ± 0.2023

#256ICA
# LOSO Accuracy: 0.2716 ± 0.3826
# LOSO Recall:   0.0440 ± 0.1982
# LOSO F1:       0.0458 ± 0.2030

# 256FICA fixed code
# LOSO Accuracy: 0.3036 ± 0.3462
# LOSO Recall:   0.0800 ± 0.2099
# LOSO F1:       0.1017 ± 0.2424

#RAW Fixed:
# LOSO Accuracy: 0.4785 ± 0.0981
# LOSO Recall:   0.1742 ± 0.1921
# LOSO F1:       0.2515 ± 0.2764

# 256FICA fixed code, small epochs
# LOSO Accuracy: 0.4632 ± 0.0606
# LOSO Recall:   0.6602 ± 0.2347
# LOSO F1:       0.5480 ± 0.1083

# 256FICA fixed code, small epochs, pca, fixed ICA, rbf,  
# LOSO Accuracy: 0.5536 ± 0.1028
# LOSO Recall:   0.9103 ± 0.2496
# LOSO F1:       0.6569 ± 0.1872

# 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# 256FICA fixed code, small epochs, lda, fixed ICA, Linear,  
# LOSO Accuracy: 0.5413 ± 0.1006
# LOSO Recall:   0.9639 ± 0.0157
# LOSO F1:       0.6903 ± 0.0898


In [ ]:
# from sklearn.model_selection import LeaveOneGroupOut
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
# import numpy as np

# # Assuming you already have:
# # allX  -> concatenated feature matrix (n_epochs_total × n_features)
# # allYLabels -> original continuous labels
# # groups -> array of participant IDs, same length as allYLabels

# y_discrete = (allYLabels >= 5).astype(int)

# logo = LeaveOneGroupOut()

# accuracies, recalls, f1s = [], [], []

# for train_idx, test_idx in tqdm(logo.split(allX, y_discrete, groups=participants)):
#     X_train, X_test = allX[train_idx], allX[test_idx]
#     y_train, y_test = y_discrete[train_idx], y_discrete[test_idx]

#     # Standardize (fit only on train)
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)

#     # Train model
#     clf = SVC(C=10, kernel='rbf', gamma='scale')
#     clf.fit(X_train, y_train)

#     # Evaluate
#     y_pred = clf.predict(X_test)
#     accuracies.append(accuracy_score(y_test, y_pred))
#     recalls.append(recall_score(y_test, y_pred))
#     f1s.append(f1_score(y_test, y_pred))

# # Report mean ± std across participants
# print(f"LOSO Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
# print(f"LOSO Recall:   {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
# print(f"LOSO F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# #Raw
# #LOSO Accuracy: 0.2785 ± 0.3908
# # LOSO Recall:   0.0429 ± 0.1966
# # LOSO F1:       0.0441 ± 0.2023

# #256ICA
# # LOSO Accuracy: 0.2716 ± 0.3826
# # LOSO Recall:   0.0440 ± 0.1982
# # LOSO F1:       0.0458 ± 0.2030

# # 256FICA fixed code
# # LOSO Accuracy: 0.3036 ± 0.3462
# # LOSO Recall:   0.0800 ± 0.2099
# # LOSO F1:       0.1017 ± 0.2424

# #RAW Fixed:
# # LOSO Accuracy: 0.4785 ± 0.0981
# # LOSO Recall:   0.1742 ± 0.1921
# # LOSO F1:       0.2515 ± 0.2764